# 01 - EDA: Telco Customer Churn

**Goal for today:** load the data, confirm it read correctly, and get a first read on
its shape, types, and target balance.

**Why this matters for MLA-C01 (Domain 1 - Data Preparation for ML):**
Before you touch a model, the exam expects you to know how to *inspect and validate*
a dataset - identify data types, spot missing values, and understand class balance.
On AWS this same step is often done with **SageMaker Data Wrangler** or a
**SageMaker Processing job**, but the underlying logic is identical to what
we're doing here with pandas.

## Step 1: Imports and load the CSV

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)  # show all columns, not truncated

df = pd.read_csv('../data/telco_churn.csv')

df.shape

(7043, 21)

`df.shape` gives you `(rows, columns)`. You should see `(7043, 21)`.

If your number of rows looks off, that's usually a sign of a parsing issue
(e.g. a stray comma inside a field) - always sanity check shape right after loading.

## Step 2: Peek at the data

**What this cell does:** `df.head()` is a DataFrame *method* (note the parentheses -
that's how you tell a method apart from an attribute like `df.shape`). By default it
returns the first 5 rows of your DataFrame, all columns included (thanks to the
`display.max_columns` setting from Step 1).

This is your first real look at the actual values, not just counts - useful for
sanity-checking that columns contain what their names suggest (e.g. does `gender`
actually contain 'Male'/'Female'? does `MonthlyCharges` look like a dollar amount?).
You can pass a number in, like `df.head(10)`, to see more rows.

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 3: Data types

This is the important one today. Pandas infers a type for every column when it reads
the CSV. Sometimes it gets it wrong - and that's a data quality issue you're expected
to catch.

**What this cell does:** `df.dtypes` is another attribute (no parentheses). It returns
a list of every column paired with the data type pandas assigned it when reading the CSV.

The main types you'll see here:
- `object` - text/string data (or a mix of types pandas couldn't pin down to one thing)
- `int64` - whole numbers
- `float64` - decimal numbers

Pandas guesses these automatically based on what's in the column. It's usually right,
but it has no idea what a column is *supposed* to represent - it only looks at the
values. That's exactly how a numeric-looking column can accidentally end up as text,
which is what we're about to see.

In [3]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Take a look at the output above. Most columns are `object` (text) which makes sense -
things like `gender`, `Contract`, `PaymentMethod` are categories.

**But look closely at `TotalCharges`.** It's `object` (text), not a number - even though
it clearly represents a dollar amount. That's a red flag. We are NOT fixing it today -
just noting it. We'll dig into *why* it's text and fix it in the next session
(spoiler: it's almost always because of blank/whitespace values hiding in numeric-looking
columns, which is a classic exam scenario for Domain 1).

## Step 4: How balanced is our target column (`Churn`)?

**What this cell does:** `df['Churn']` first selects a single column out of the
DataFrame - square brackets with a column name in quotes pulls out that one column as
a **Series** (pandas' term for a single column of data, essentially a 1-column version
of a DataFrame).

`.value_counts()` then counts how many times each unique value appears in that Series.
Since `Churn` only contains 'Yes' and 'No', this tells you exactly how many customers
churned vs. didn't - the raw counts, not percentages yet.

In [4]:
df['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

**What this cell does:** Same idea, but with two additions:
- `normalize=True` tells `.value_counts()` to return **proportions** (fractions that
  add up to 1.0) instead of raw counts
- `* 100` converts those fractions into percentages, which are easier to read at a glance

This is the same underlying method, just parameterized differently - a good habit to
notice, since most pandas methods have optional arguments like `normalize` that change
the shape of the output without changing what the method fundamentally does.

In [5]:
df['Churn'].value_counts(normalize=True) * 100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

Take note of this percentage split. If one class heavily outnumbers the other
(e.g. 90/10), that's called **class imbalance**, and it directly affects:
- which evaluation metric you should trust (accuracy becomes misleading)
- whether you need techniques like SMOTE, class weighting, or resampling

This dataset has a *moderate* imbalance (roughly 73/27) - not extreme, but enough
that it matters. We'll come back to this explicitly when we choose our evaluation
metric in a later session (this maps to Domain 2 - Model Development).

---

**That's it for today.** Small, complete increment:
- loaded the data
- confirmed shape
- inspected dtypes and flagged `TotalCharges` as suspicious
- checked target balance

**Next session:** we'll investigate `TotalCharges`, check for missing values across
all columns properly, and start looking at individual feature distributions.